# PixelVAR Final Demo Notebook

**5-minute story:** a sprite becomes transparent palette tokens, PixelVAR generates in that token space, and the audit explains why raw FID alone is not enough.

This notebook is designed for a live demo on Lightning AI. Most cells use saved project artifacts, while the live sampling cell runs only if the checkpoint and processed palette are available locally.

In [ ]:
# Colab bootstrap: run this first if you opened the notebook from GitHub/Colab.
# Colab loads the notebook file, but it does not automatically clone the whole repo.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/mahirgit/PixelVAR.git"
BRANCH = "codex/pixelvar-external-baselines"
PROJECT_ROOT = Path("/content/PixelVAR")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print(f"Cloning {REPO_URL} ...")
        subprocess.check_call(["git", "clone", REPO_URL, str(PROJECT_ROOT)])
    else:
        print("Repo already exists; fetching latest branch...")
        subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"])

    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "origin", BRANCH])

    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {Path.cwd()}")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "-q"])
else:
    print("Not running in Colab. Continuing with the local repo.")

## Demo Run Order

| Time | Section | What to show |
|---:|---|---|
| 0:00-0:45 | Representation | RGBA sprite, alpha mask, 16-color palette, token map, six-scale pyramid |
| 0:45-1:45 | Live or fallback generation | Generate a small PixelVAR grid, or show the saved selected-run grid |
| 1:45-2:25 | Sampling control | Temperature/top-k changes output style |
| 2:25-3:15 | Internal branches | PixelVAR vs HMAR vs generated/mixed/Patch-VQ branches |
| 3:15-4:20 | Metric trap | Raw FID picks Flat AR, exact-match audit changes the decision |
| 4:20-5:00 | External baselines + scale | Diffusion-style rows fail normalized 32x32 protocol; 170K pass stayed stable |

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import sys
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image as PILImage
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def find_project_root(start: Path | None = None) -> Path:
    """Find the PixelVAR repo root from a notebook, script, or Lightning working directory."""
    candidates = []
    if start is not None:
        candidates.append(start.resolve())
    candidates.append(Path.cwd().resolve())
    candidates.extend(Path.cwd().resolve().parents)
    for extra in [
        Path('/teamspace/studios/this_studio/PixelVAR'),
        Path('/teamspace/studios/this_studio'),
        Path('/content/PixelVAR'),
    ]:
        candidates.append(extra)
    for candidate in candidates:
        if (candidate / 'setup.py').exists() and (candidate / 'pixelvar').exists():
            return candidate
    raise RuntimeError('Could not find PixelVAR repo root. Open this notebook from inside the repo.')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEMO_OUT = ROOT / 'outputs' / 'final_demo_notebook'
DEMO_OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.facecolor': '#111111',
    'figure.facecolor': '#111111',
    'savefig.facecolor': '#111111',
    'text.color': '#f5f0e8',
    'axes.labelcolor': '#d7dee8',
    'xtick.color': '#d7dee8',
    'ytick.color': '#d7dee8',
})


def read_csv_rows(path: Path) -> list[dict]:
    with Path(path).open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def as_float(value, default=float('nan')):
    try:
        return float(value)
    except Exception:
        return default


def as_int(value, default=0):
    try:
        return int(float(value))
    except Exception:
        return default


def markdown_table(rows: list[dict], columns: list[str], formats: dict[str, str] | None = None, title: str | None = None):
    formats = formats or {}
    lines = []
    if title:
        lines.append(f'**{title}**')
        lines.append('')
    lines.append('| ' + ' | '.join(columns) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(columns)) + ' |')
    for row in rows:
        vals = []
        for col in columns:
            value = row.get(col, '')
            fmt = formats.get(col)
            if fmt and isinstance(value, (int, float)) and not math.isnan(float(value)):
                vals.append(fmt.format(value))
            else:
                vals.append(str(value))
        lines.append('| ' + ' | '.join(vals) + ' |')
    display(Markdown('\n'.join(lines)))

print(f'Project root: {ROOT}')
print(f'Demo outputs: {DEMO_OUT}')

## 1. The Representation: A Sprite Is A Palette-Token Pyramid

This is the core idea to show first: PixelVAR does not generate continuous RGB and then repair it. The model generates token IDs: `0` for transparency and `1..16` for palette colors. The same token map is viewed at six scales.

In [ ]:
from pixelvar.tokenizers import DeterministicPyramidTokenizer


def checkerboard_composite(rgba: np.ndarray, cell: int = 4) -> np.ndarray:
    rgba = np.asarray(rgba).astype(np.uint8)
    h, w = rgba.shape[:2]
    yy, xx = np.indices((h, w))
    checks = ((yy // cell + xx // cell) % 2).astype(np.uint8)
    bg = np.where(checks[..., None] == 0, 235, 205).astype(np.uint8)
    alpha = rgba[..., 3:4].astype(np.float32) / 255.0
    comp = rgba[..., :3].astype(np.float32) * alpha + bg.astype(np.float32) * (1.0 - alpha)
    return comp.astype(np.uint8)


def extract_first_generated_png() -> Path | None:
    direct = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_8192' / 'images' / 'sample_000000.png'
    if direct.exists():
        return direct
    package = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_8192.zip'
    if not package.exists():
        return None
    target = DEMO_OUT / 'sample_000000.png'
    if not target.exists():
        with zipfile.ZipFile(package) as zf:
            with zf.open('images/sample_000000.png') as src:
                target.write_bytes(src.read())
    return target


def quantize_rgba_to_demo_tokens(rgba: np.ndarray, max_colors: int = 16) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fallback quantizer for demo visualization when the original processed palette is absent."""
    rgba = np.asarray(rgba).astype(np.uint8)
    alpha_mask = rgba[..., 3] >= 128
    rgb = PILImage.fromarray(rgba[..., :3], mode='RGB')
    quantized = rgb.quantize(colors=max_colors, method=PILImage.Quantize.MEDIANCUT, dither=PILImage.Dither.NONE)
    palette_values = np.array(quantized.getpalette(), dtype=np.uint8).reshape(-1, 3)
    used = np.unique(np.asarray(quantized)[alpha_mask]) if alpha_mask.any() else np.array([0], dtype=np.uint8)
    used_count = max(int(used.max()) + 1 if len(used) else 1, min(max_colors, len(palette_values)))
    palette_raw = palette_values[:used_count]
    if len(palette_raw) < max_colors:
        pad_color = palette_raw[-1:] if len(palette_raw) else np.array([[0, 0, 0]], dtype=np.uint8)
        palette_raw = np.concatenate([palette_raw, np.repeat(pad_color, max_colors - len(palette_raw), axis=0)], axis=0)
    else:
        palette_raw = palette_raw[:max_colors]
    q = np.asarray(quantized).astype(np.uint8)
    q = np.clip(q, 0, max_colors - 1)
    index_map = np.zeros(q.shape, dtype=np.uint8)
    index_map[alpha_mask] = q[alpha_mask] + 1
    rendered = np.zeros((*q.shape, 4), dtype=np.uint8)
    rendered[alpha_mask, :3] = palette_raw[q[alpha_mask]]
    rendered[alpha_mask, 3] = 255
    return index_map, palette_raw, rendered


def load_demo_sprite_and_tokens():
    processed_dir = ROOT / 'data' / 'processed' / 'sprites'
    index_path = processed_dir / 'index_maps.npy'
    palette_path = processed_dir / 'palette.json'
    if index_path.exists() and palette_path.exists():
        from pixelvar.data.palette import PaletteExtractor
        palette = PaletteExtractor().load(palette_path)
        index_map = np.load(index_path, mmap_mode='r')[0].astype(np.uint8)
        rendered = palette.render_index_map(index_map)
        return rendered, index_map, palette.palette, 'processed training sprite'

    img_path = extract_first_generated_png()
    if img_path is None:
        raise FileNotFoundError('No processed sprites or generated sample PNGs found for the representation demo.')
    rgba = np.array(PILImage.open(img_path).convert('RGBA'))
    index_map, palette, rendered = quantize_rgba_to_demo_tokens(rgba)
    return rendered, index_map, palette, 'saved generated sample, locally quantized for demo display'


def token_rgb(index_map: np.ndarray, palette: np.ndarray) -> np.ndarray:
    tokens = np.asarray(index_map)
    rgb = np.zeros((*tokens.shape, 3), dtype=np.uint8)
    mask = tokens > 0
    rgb[~mask] = [18, 18, 18]
    rgb[mask] = palette[np.clip(tokens[mask] - 1, 0, len(palette) - 1)]
    return rgb

sprite_rgba, index_map, palette, source = load_demo_sprite_and_tokens()
tokenizer = DeterministicPyramidTokenizer([1, 2, 4, 8, 16, 32])
scale_maps = tokenizer.encode(index_map)
sequence = tokenizer.to_sequence(scale_maps)

fig = plt.figure(figsize=(13, 7))
grid = fig.add_gridspec(3, 6, height_ratios=[1.05, 0.35, 1.0], hspace=0.35, wspace=0.25)

ax = fig.add_subplot(grid[0, 0])
ax.imshow(checkerboard_composite(sprite_rgba))
ax.set_title('RGBA sprite')
ax.axis('off')

ax = fig.add_subplot(grid[0, 1])
ax.imshow(index_map == 0, cmap='gray', interpolation='nearest')
ax.set_title('transparent token 0')
ax.axis('off')

ax = fig.add_subplot(grid[0, 2])
ax.imshow(token_rgb(index_map, palette), interpolation='nearest')
ax.set_title('palette token map')
ax.axis('off')

swatch = np.zeros((24, 16 * 24, 3), dtype=np.uint8)
for i, color in enumerate(palette[:16]):
    swatch[:, i * 24 : (i + 1) * 24] = color
ax = fig.add_subplot(grid[0, 3:6])
ax.imshow(swatch, interpolation='nearest')
ax.set_title('tokens 1..16: palette colors')
ax.set_xticks([i * 24 + 12 for i in range(16)])
ax.set_xticklabels([str(i + 1) for i in range(16)], fontsize=8)
ax.set_yticks([])

for i, scale_map in enumerate(scale_maps):
    ax = fig.add_subplot(grid[2, i])
    ax.imshow(token_rgb(np.asarray(scale_map), palette), interpolation='nearest')
    res = tokenizer.scale_resolutions[i]
    ax.set_title(f'{res}x{res}\n{res*res} tokens', fontsize=9)
    ax.axis('off')

fig.suptitle(f'PixelVAR representation: 0 transparent + 16 palette tokens -> 1,365-token pyramid\nsource: {source}', fontsize=13, y=0.99)
plt.show()
print(f'Token sequence shape: {tuple(sequence.shape)}; first/last scales: {tokenizer.scale_resolutions[0]}x{tokenizer.scale_resolutions[0]} -> {tokenizer.scale_resolutions[-1]}x{tokenizer.scale_resolutions[-1]}')

## 2. Live Mini Generation With Safe Fallback

Run this during the demo if the checkpoint and processed palette are available. If not, the cell automatically displays the saved selected-run grid. This keeps the demo reliable on Lightning.

In [ ]:
def show_image_file(path: Path, title: str, figsize=(7, 5)):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    img = PILImage.open(path).convert('RGBA')
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(checkerboard_composite(np.array(img)))
    ax.set_title(title)
    ax.axis('off')
    plt.show()


def try_live_pixelvar_generation(num_samples: int = 32, temperature: float = 0.8, top_k: int = 8):
    checkpoint = ROOT / 'modal_checkpoints' / 'var_sprites_v0_full_best.ckpt'
    config_path = ROOT / 'configs' / 'train' / 'sprites_v0_full.yaml'
    fallback = ROOT / 'reports' / 'eval' / 'sprites_v0_full' / 'temp_0.8_topk_8_grid.png'

    if not checkpoint.exists():
        return fallback, f'Fallback: checkpoint not found at {checkpoint}'
    if not config_path.exists():
        return fallback, f'Fallback: config not found at {config_path}'

    try:
        import torch
        from pixelvar.data.palette import PaletteExtractor
        from pixelvar.training import load_var_model_from_checkpoint
        from pixelvar.utils import load_yaml, save_rgba_grid, tokens_to_rgba
    except Exception as exc:
        return fallback, f'Fallback: could not import inference stack: {exc}'

    config = load_yaml(config_path)
    processed_dir = ROOT / config['data']['processed_dir']
    palette_path = processed_dir / 'palette.json'
    if not palette_path.exists():
        return fallback, f'Fallback: processed palette missing at {palette_path}. Saved grid is from the same selected checkpoint.'

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(7)
    if device == 'cuda':
        torch.cuda.manual_seed_all(7)

    try:
        model = load_var_model_from_checkpoint(checkpoint, map_location=device).to(device)
        with torch.no_grad():
            tokens = model.sample(batch_size=num_samples, temperature=temperature, top_k=top_k, device=device).cpu()
        palette = PaletteExtractor().load(palette_path)
        images = tokens_to_rgba(tokens, palette, scale_resolutions=config['model'].get('scale_resolutions'))
        out = DEMO_OUT / f'live_pixelvar_t{temperature}_top{top_k}_{num_samples}.png'
        save_rgba_grid(images, out, columns=8)
        return out, f'Live generation succeeded on {device}: {num_samples} samples, temp={temperature}, top_k={top_k}'
    except Exception as exc:
        return fallback, f'Fallback: live generation failed safely: {type(exc).__name__}: {exc}'

live_grid, live_message = try_live_pixelvar_generation(num_samples=32, temperature=0.8, top_k=8)
display(Markdown(f'**{live_message}**'))
show_image_file(live_grid, 'PixelVAR selected setting: temperature=0.8, top_k=8', figsize=(7.5, 5.0))

## 3. Sampling Control: Same Checkpoint, Different Sampling Settings

This section is quick but visually useful: sampling controls change style and diversity without changing the trained model.

In [ ]:
sweep = [
    ('temp=0.6, top_k=8\nmore conservative', ROOT / 'reports' / 'eval' / 'sprites_v0_full' / 'temp_0.6_topk_8_grid.png'),
    ('temp=0.8, top_k=8\nselected setting', ROOT / 'reports' / 'eval' / 'sprites_v0_full' / 'temp_0.8_topk_8_grid.png'),
    ('temp=1.0, top_k=8\nmore varied', ROOT / 'reports' / 'eval' / 'sprites_v0_full' / 'temp_1_topk_8_grid.png'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (title, path) in zip(axes, sweep):
    if path.exists():
        ax.imshow(checkerboard_composite(np.array(PILImage.open(path).convert('RGBA'))))
    else:
        ax.text(0.5, 0.5, f'Missing:\n{path.name}', ha='center', va='center')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle('Sampling controls change the look without retraining', fontsize=13)
plt.show()

## 4. Internal Branches: The Main Model Was Chosen Against Real Alternatives

The demo should make clear that we implemented more than one model path. PixelVAR is selected because it is the strongest completed comparable branch, not because it was the only branch.

In [ ]:
decision_path = ROOT / 'reports' / 'final' / 'model_decision_table.csv'
decision = read_csv_rows(decision_path)
rows = []
for row in decision[:6]:
    rows.append({
        'rank': row['rank'],
        'branch': row['branch'],
        'run': row['run'],
        'best_score': as_float(row['best_score']),
        'temperature': row['temperature'],
        'top_k': row['top_k'],
        'refinement_steps': row['refinement_steps'] or '-',
        'decision': row['decision'],
    })
markdown_table(rows, ['rank', 'branch', 'run', 'best_score', 'temperature', 'top_k', 'refinement_steps', 'decision'], {'best_score': '{:.5f}'})

branch_sheet = ROOT / 'reports' / 'final' / 'final_branch_comparison_sheet.png'
if branch_sheet.exists():
    show_image_file(branch_sheet, 'Branch sample comparison: PixelVAR, HMAR, generated/mixed variants, Patch-VQ', figsize=(11.5, 6.0))
else:
    display(Markdown(f'Branch sample sheet missing: `{branch_sheet}`'))

## 5. The Metric Trap: Raw FID Rewards Memorization

This is the most important scientific moment in the demo. First show the raw metric winner, then reveal the exact-match audit.

In [ ]:
known_rows = read_csv_rows(ROOT / 'reports' / 'final' / 'known_metrics_comparison.csv')
name_map = {
    'pixelvar_main': 'PixelVAR main',
    'hmar_steps1': 'HMAR step=1',
    'flat_ar': 'Flat AR',
    'flat_maskgit': 'Flat MaskGIT',
}
audit = {
    'pixelvar_main': {'train_exact': 0, 'val_exact': 13, 'test_exact': 11, 'audit_read': 'keep, main result'},
    'hmar_steps1': {'train_exact': 0, 'val_exact': 15, 'test_exact': 22, 'audit_read': 'keep, close'},
    'flat_ar': {'train_exact': 3162, 'val_exact': 365, 'test_exact': 333, 'audit_read': 'memorizing'},
    'flat_maskgit': {'train_exact': 0, 'val_exact': 0, 'test_exact': 0, 'audit_read': 'weak coverage'},
}
scoreboard = []
for row in known_rows:
    method = row['method']
    item = {
        'method': method,
        'label': name_map.get(method, method),
        'fid': as_float(row['fid']),
        'kid_mean': as_float(row['kid_mean']),
        'precision': as_float(row['precision']),
        'recall': as_float(row['recall']),
        'coverage': as_float(row['coverage']),
    }
    item.update(audit.get(method, {}))
    scoreboard.append(item)

markdown_table(
    scoreboard,
    ['label', 'fid', 'kid_mean', 'precision', 'recall', 'coverage', 'val_exact', 'test_exact', 'audit_read'],
    {'fid': '{:.2f}', 'kid_mean': '{:.5f}', 'precision': '{:.3f}', 'recall': '{:.3f}', 'coverage': '{:.3f}'},
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
colors = ['#45c26b' if r['method'] == 'pixelvar_main' else '#62c7da' if r['method'] == 'hmar_steps1' else '#ff5c7a' if r['method'] == 'flat_ar' else '#f0b84f' for r in scoreboard]
labels = [r['label'] for r in scoreboard]
axes[0].bar(labels, [r['fid'] for r in scoreboard], color=colors)
axes[0].set_title('Raw FID: lower is better')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', color='#333333', alpha=0.4)

x = np.arange(len(scoreboard))
axes[1].bar(x - 0.18, [r['val_exact'] for r in scoreboard], width=0.36, label='validation exact', color='#ff5c7a')
axes[1].bar(x + 0.18, [r['test_exact'] for r in scoreboard], width=0.36, label='test exact', color='#f0b84f')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=20)
axes[1].set_title('Exact-match audit: lower is better')
axes[1].legend(frameon=False)
axes[1].grid(axis='y', color='#333333', alpha=0.4)
plt.tight_layout()
plt.show()

display(Markdown('**Demo line:** Raw FID picks Flat AR, but the audit exposes exact reproduction. PixelVAR is selected because it is strong without that memorization failure.'))

## 6. External Baselines: Practical Pressure Tests

These are not claimed as perfect apples-to-apples SOTA comparisons. The point is practical: under the same normalized 32x32 RGBA protocol, accessible diffusion-style outputs did not match the sprite-token model.

In [ ]:
external = [
    {'method': 'PixelVAR', 'n': 256, 'fid': 46.48, 'coverage': 0.891, 'read': 'selected palette-token model'},
    {'method': 'Pokemon LoRA', 'n': 256, 'fid': 154.02, 'coverage': 0.051, 'read': 'best practical external visual row'},
    {'method': 'SSD-1B', 'n': 256, 'fid': 158.58, 'coverage': 0.035, 'read': 'recognizable sometimes, low precision'},
    {'method': 'SD-piXL', 'n': 16, 'fid': 497.71, 'coverage': 0.000, 'read': 'targeted attempt failed protocol'},
]
markdown_table(external, ['method', 'n', 'fid', 'coverage', 'read'], {'fid': '{:.2f}', 'coverage': '{:.3f}'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ext_colors = ['#45c26b', '#f0b84f', '#ff7f1f', '#ff405e']
methods = [r['method'] for r in external]
axes[0].bar(methods, [r['fid'] for r in external], color=ext_colors)
axes[0].set_title('External FID: lower is better')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', color='#333333', alpha=0.4)
axes[1].bar(methods, [r['coverage'] for r in external], color=ext_colors)
axes[1].set_title('External coverage: higher is better')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', color='#333333', alpha=0.4)
plt.tight_layout()
plt.show()

sample_rows = [
    ('PixelVAR main', ROOT / 'reports' / 'final' / 'final_main_var_sample_sheet.png'),
    ('Pokemon LoRA', ROOT / 'reports' / 'final' / 'pokemon_sprite_lora_sample_sheet.png'),
    ('SSD-1B', ROOT / 'reports' / 'final' / 'practical_diffusion_sample_sheet.png'),
    ('SD-piXL', ROOT / 'reports' / 'final' / 'sd_pixl_sample_sheet.png'),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (title, path) in zip(axes.reshape(-1), sample_rows):
    if path.exists():
        ax.imshow(checkerboard_composite(np.array(PILImage.open(path).convert('RGBA'))))
    else:
        ax.text(0.5, 0.5, f'Missing:\n{path.name}', ha='center', va='center')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('External visual rows after normalized sprite evaluation', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Scale Check And Final Takeaway

End the demo with the large generation pass. Do not call this a human approval rate; it is an automatic stability gate.

In [ ]:
summary_path = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_170000' / 'summary.json'
inspection_path = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_170000_inspection' / 'inspection_summary.json'

summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
inspection = json.loads(inspection_path.read_text()) if inspection_path.exists() else {}

scale_cards = [
    {'quantity': 'generated samples', 'value': int(summary.get('num_samples', 170000))},
    {'quantity': 'automatic keep', 'value': int(inspection.get('keep_count', 161479))},
    {'quantity': 'review bucket', 'value': int(inspection.get('review_count', 8500))},
    {'quantity': 'reject bucket', 'value': int(inspection.get('reject_count', 21))},
]
markdown_table(scale_cards, ['quantity', 'value'])

fig, ax = plt.subplots(figsize=(7.5, 3.6))
vals = [scale_cards[i]['value'] for i in [1, 2, 3]]
labels = ['automatic keep', 'review', 'reject']
ax.bar(labels, vals, color=['#45c26b', '#f0b84f', '#ff5c7a'])
ax.set_title('170K generation pass: automatic gate counts')
ax.grid(axis='y', color='#333333', alpha=0.4)
for i, v in enumerate(vals):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', color='#f5f0e8')
plt.show()

keep_grid = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_170000_inspection' / 'grids' / 'keep_random.png'
reject_grid = ROOT / 'reports' / 'generated' / 'sprites_v0_full_t08_top8_170000_inspection' / 'grids' / 'rejects.png'
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, title, path in [
    (axes[0], 'random automatic-keep samples', keep_grid),
    (axes[1], 'rare reject bucket', reject_grid),
]:
    if path.exists():
        ax.imshow(checkerboard_composite(np.array(PILImage.open(path).convert('RGBA'))))
    else:
        ax.text(0.5, 0.5, f'Missing:\n{path.name}', ha='center', va='center')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

display(Markdown('### Final demo line\nPixelVAR is a bounded but defensible 32x32 transparent-sprite generator because the representation matches the domain, and the evaluation is memorization-aware.'))

## Lightning AI Commands

Run these from a fresh Lightning terminal if needed:

```bash
git clone https://github.com/mahirgit/PixelVAR.git
cd PixelVAR
git checkout codex/pixelvar-external-baselines
python -m pip install -r requirements.txt
python -m pip install -e .
jupyter lab notebooks/02_pixelvar_final_demo.ipynb
```

If the live generation cell falls back, the demo is still valid: it displays saved outputs generated from the selected `var_sprites_v0_full` checkpoint.